In [ ]:
import os
import re
import numpy as np
import matplotlib

from data_loader import load_masked_channel_data

from preprocess import  prepare_data_concatenated
from model import detect_anomalies_isolation_forest, perform_kmeans_clustering, create_cluster_mask
from visualization import plot_results
import matplotlib.patches as mpatches
import sunpy.map

from typing import List, Tuple, Set, Union

from skimage.transform import resize



# --- Configuration ---
config = {
    "data_dir": "testing_input",
    "anomaly_thresholds": [0.1],
    "output_dir": "./output_figures",
    "image_size": 512,
    "contamination": 0.05,
    "n_clusters": 7,
    "max_k": 10,
    "random_state": 42
}

In [7]:
masked_data_list, channel_names, valid_files = load_masked_channel_data(
    config["data_dir"],
    config["image_size"]
)

In [ ]:


def run_pipeline(config):
    """
    Runs the entire pipeline for anomaly detection and clustering on solar image data.

    Args:
        config (dict): A dictionary containing configuration settings, including data directory, 
                       channels, anomaly thresholds, and other parameters for clustering and 
                       anomaly detection.

    Returns:
        None: The function saves the results as output files in the specified directory.
    """


# --- 0. Create output directory if it doesn't exist ---
os.makedirs(config["output_dir"], exist_ok=True)

# --- 1. Load and Preprocess Data ---
masked_data_list, channel_names, valid_files = load_masked_channel_data(
    config["data_dir"],
    config["image_size"]
)

# --- 2. Prepare data for anomaly detection ---
prepared_data, valid_pixel_mask, nan_mask = prepare_data_concatenated(masked_data_list)


# --- 3. Anomaly detection with Isolation Forest ---
anomaly_map = detect_anomalies_isolation_forest(prepared_data, config["contamination"],config["image_size"],valid_pixel_mask)


# --- 4. Anomaly thresholding and clustering ---
# For each threshold, compute anomaly mask, extract features, perform clustering, and visualize results

for threshold in config["anomaly_thresholds"]:
    print(f"\nProcessing with anomaly threshold: {threshold}")

    # Identify pixels considered anomalous
    anomaly_mask = anomaly_map < threshold
    num_anomalies = np.sum(anomaly_mask)
    print(f"Anomalies detected: {num_anomalies}")

    if num_anomalies == 0:
        print("No anomalies detected for this threshold.")
        cluster_mask = np.zeros_like(anomaly_mask, dtype=int)
        cluster_cmap = matplotlib.colors.ListedColormap([])
        cluster_patches = []
        n_clusters = 0
    else:
        # Flatten nan mask and get valid pixel indices once
        valid_indices = np.argwhere(~nan_mask.reshape((config["image_size"], config["image_size"])))
        index_map = {tuple(idx): i for i, idx in enumerate(valid_indices)}

        # Get the positions of anomalous pixels that are valid
        anomaly_indices = np.argwhere(anomaly_mask)
        valid_keys = [tuple(idx) for idx in anomaly_indices if tuple(idx) in index_map]

        if not valid_keys:
            print("  No valid anomaly pixels found within the valid pixel mask.")
            cluster_mask = np.zeros_like(anomaly_mask, dtype=int)
            cluster_cmap = matplotlib.colors.ListedColormap([])
            cluster_patches = []
            n_clusters = 0
        else:
            # Retrieve features and positions for valid anomaly pixels
            feature_indices = [index_map[key] for key in valid_keys]
            features = prepared_data[feature_indices]
            valid_pixel_positions = np.array(valid_keys)

            # Perform KMeans clustering on the anomaly features
            cluster_labels, _ = perform_kmeans_clustering(
                features,
                config["n_clusters"],
                random_state=config["random_state"]
            )
            print(f"Clusters formed: {np.unique(cluster_labels).size}")

            # Create cluster mask and color map
            cluster_mask, cluster_cmap, cluster_patches, n_clusters = create_cluster_mask(
                anomaly_mask,
                cluster_labels,
                valid_pixel_mask,
                config["image_size"]
            )

    # --- 4.1. Visualization ---
    plot_results(
        masked_data_list,
        cluster_mask,
        cluster_cmap,
        n_clusters,
        cluster_patches,
        channel_names,
        threshold,
        config["output_dir"]
    )


In [ ]:

# --- Run pipeline ---
if __name__ == "__main__":
    run_pipeline(config)
